## 04. scratchpad

### 1. Data Upload

In [42]:
import pandas as pd
import os
from collections import defaultdict 
import re

In [43]:
def upload_to_dataframe(root, files, num_lines):
    path_eng, path_pol = [root+f for f in files]
    data = defaultdict(list)
    
    with open (path_eng, encoding="utf_8") as f_eng, open (path_pol, encoding="utf_8") as f_pol:
        for _ in range(num_lines):
            data['eng_text'].append(f_eng.readline().strip())
            data['pol_text'].append(f_pol.readline().strip())
    return pd.DataFrame(data)

In [44]:
root = "../data/opus_opensub/en-pl.txt/"
files = ('OpenSubtitles.en-pl.en', '/OpenSubtitles.en-pl.pl')
df = upload_to_dataframe(root, files, 600000)

### 2. EDA & Sanitazation

#### 2.1 English Non-Ascii Sentences

In [45]:
def nonascii_list(df_series, is_pol):
    if is_pol:
        pl_chars = set("ąęćłńóśźżĄĘĆŁŃÓŚŹŻ")
        requir = lambda x: x.isascii() or x in pl_chars
    else:
        requir = lambda x: x.isascii()
        
    text_all = ''.join(df_series)
    unique_dct = {x for x in text_all if not requir(x)}
    return sorted(list(unique_dct))

In [46]:
print(nonascii_list(df['eng_text'], False))

['\x80', '\x81', '\x82', '\x83', '\x84', '\x85', '\x88', '\x8b', '\x8c', '\x8e', '\x8f', '\x91', '\x94', '\x96', '\x98', '\x99', '\x9c', '\x9d', '\xa0', '¡', '¢', '£', '¤', '¥', '¦', '§', '¨', '©', 'ª', '¬', '\xad', '°', '±', '²', '³', '´', 'µ', '¶', '·', '¸', '¹', 'º', '¼', '½', '¾', '¿', 'Á', 'Â', 'Ã', 'Ä', 'Å', 'Ç', 'É', 'Ê', 'Ì', 'Ï', 'Ð', 'Ñ', 'Ô', 'Ö', 'Ø', 'Ü', 'Ý', 'Þ', 'ß', 'à', 'á', 'â', 'ã', 'ä', 'å', 'ç', 'è', 'é', 'ê', 'ì', 'í', 'î', 'ï', 'ð', 'ñ', 'ó', 'ô', 'ö', 'ù', 'ú', 'û', 'ü', 'ý', 'þ', 'Ă', 'ď', 'ę', 'Ł', 'ń', 'ő', 'œ', 'ś', 'Š', 'š', 'Ј', 'С', 'й', '،', 'أ', 'إ', 'ئ', 'ا', 'ب', 'ة', 'ت', 'ح', 'ر', 'س', 'ش', 'ع', 'غ', 'ف', 'ك', 'ل', 'م', 'ن', 'و', 'ي', 'َ', 'ُ', 'ِ', 'ّ', 'ْ', '\u200b', '‒', '–', '—', '‚', '€', '™', '─', '☻', '♥', '♪', '慹', '拢', '檛', '鈥', 'ﬁ', 'ﬂ']


In [47]:
is_ascii = lambda text: text.isascii()
maska = df['eng_text'].apply(is_ascii)
df_non_ascii = df[~maska].copy().reset_index(drop=True)
print(df_non_ascii.shape)
df_non_ascii.head()

(4595, 2)


,eng_text,pol_text
0,¿You had never seen A crab suedes? Them there ...,- Nigdy wcześniej nie widziałaś raków?
1,"He hears ¿, where you live?",Nigdy wcześniej nie byłam w lesie.
2,"Boys ¿, they are there?",Jesteście tam chłopaki?
3,- Te atrapé. - No es gracioso. Your he catches...,- Ale twoje mina była.
4,"Your also ¿, it is not like that?","- Tak, niezła jest"


In [48]:
df_2 = df[maska].reset_index(drop=True)
df_2.shape

(595405, 2)

#### 2.2 Polish Non-Ascii Sentences

In [50]:
pl_chars = set("ąęćłńóśźżĄĘĆŁŃÓŚŹŻ")
requir = lambda let: not let.isascii() and let not in pl_chars
maska = df_2['pol_text'].apply(lambda snt: any([requir(let) for let in snt]))
df_non_ascii = df_2[maska].copy().reset_index(drop=True)
print(df_non_ascii.shape)
df_non_ascii.sample(10)

(1227, 2)


,eng_text,pol_text
766,Buy up every minute of every radio station in ...,Wykup každa minute w každej podleglej ci stacj...
1223,"On the blackboard is drawn, Mihailo Potapych, ...","Jest tam narysowane, Mihhailo Potapõtð, głuchy..."
224,"- You are, aren't you?",– Istotnie. Tak przy okazji:
68,The Church is opening her arms to you if you t...,Ko¶ciół wyci±ga do ciebie rękę je¶li j± odrzuc...
837,"The world is not as bad as you think, Tom.","Tom, świat nie jest taki zły jak ​​myślisz."
287,- It's nothing at all.,– Niezupełnie.
944,My Fuhrer!,Meine führer!
658,I was hoping you'd be spared all this.,"Myslalem, že ci tego oszczedze."
215,- There's no need to hide it.,"– Nao-san, mów szczerze."
735,"I accuse this man. By his tone, by his careful...","Oskaržam tego czlowieka, že umiejetnie dobiera..."


In [55]:
df_3 = df_2[~maska].reset_index(drop=True)
print(df_3.shape)
df_3.sample(5)

(594178, 2)


,eng_text,pol_text
470445,"- Oh, a lot of things. He made some crack abou...","- Wiele rzeczy żartował sobie, że jestem mamin..."
216227,- Just want a table for two.,- Po prostu chce stolik dla dwoch osob.
298936,Nearer.,Bliżej.
148899,"Just this way, please.","Tędy, proszę."
551495,"Oh, you remembered.","Och, pamiętasz."


In [60]:
print(nonascii_list(df_3['eng_text'], False))
print(nonascii_list(df_3['pol_text'], True))

[]
[]


In [67]:
df_3.iloc[74182]['eng_text']

'The next time I...'

In [68]:
df_3.iloc[74182]['pol_text']

'Następnym razem...'

#### 2.3 Short Or Empty Examples

In [154]:
mask1 = df_3['eng_text'].str.len() >= 3
mask2 = df_3['pol_text'].str.len() >= 3

In [157]:
df_4 = df_3[mask1 & mask2].reset_index(drop=True)
df_4.head()

,eng_text,pol_text
0,"Previously on ""The Blacklist""...",/W poprzednich odcinkach: /
1,- You want to call your daddy?,- Chcesz zadzwonić do taty?
2,"- Yeah, I want to tell him I'm okay.","- Tak, powiem, że wszystko w porządku."
3,Okay.,Dobrze.
4,Lizzy... Be careful of your husband.,"Lizzy, uważaj na męża."


In [158]:
df_4.shape

(593894, 2)

#### 2.4 Dialog like sentences [starts with - ]

In [162]:
is_dialog = lambda snt: bool(re.match(r"^ *- *", snt))
mask = df_4['eng_text'].apply(is_dialog) | df_4['pol_text'].apply(is_dialog)
df_dialogs = df_4[mask].copy().reset_index(drop=True)
df_dialogs.head()

,eng_text,pol_text
0,- You want to call your daddy?,- Chcesz zadzwonić do taty?
1,"- Yeah, I want to tell him I'm okay.","- Tak, powiem, że wszystko w porządku."
2,- 45 seconds.,Szybciej! 45 sekund.
3,It's okay. It's okay. It's okay.,- Już dobrze.
4,It's okay. You're okay. You're okay.,- Odpoczywaj.


In [192]:
def fix_dialog(snt):
    return re.sub(r"^ *- *", "", snt)

df_4['eng_text'] = df_4['eng_text'].apply(fix_dialog).reset_index(drop=True)
df_4['pol_text'] = df_4['pol_text'].apply(fix_dialog).reset_index(drop=True)
df_4.sample(30)

,eng_text,pol_text
583770,That's not how you do it.,Spójrz! Tak niedobrze!
455445,"Well, let's see, that's, uh...","Więc, zobaczmy, to jest..."
145011,You can't change that.,Nie można tego zmienić.
396136,For the German.,Dla Niemca.
488641,This vacuum I'm living in...,Ta próznia w której zyje...
480663,"""Till stars of evening cease to burn.""","""Aż wieczorne gwiazdy przestaną płonąć."""
392387,On account of the Oklahoma Tool Supply Company...,Ponieważ włąścicielem firmy Oklahoma Tool Supp...
107305,"Well, both.",To i to.
450431,Please don't.,"Proszę, przestań."
410804,That's a horrible thing to say.,To okropne co mówisz! Myślałaś że kim w ogóle ...


#### 2.5 Broken Dialog like sentences [somehwere has -]

In [258]:
broken_dialog = lambda snt: bool(re.match(r".* +- +.*", snt))
mask1 = df_4['eng_text'].apply(broken_dialog)
mask2 = df_4['pol_text'].apply(broken_dialog)

In [267]:
df_4[mask1 | mask2].sample(5)

,eng_text,pol_text
256144,I'm so glad you got back soon.,"Cieszę się, że tak szybko wróciłeś. - Jest wsp..."
320251,"Oh, I just don't love him. - Oh, there, there,...",Nie kocham go.
473921,We have to see Grandma Death.,"Gdzie idziemy? - Posłuchaj, musimy iść. - Gdzie?"
258947,Casing tools and lumber will cost you about $ ...,"Narzędzia i drewno do budowy, - to będzie okoł..."
233013,If I were killed... - You mustn't say that!,Gdybym zginął...


In [276]:
df_5 = df_4[~mask1 & ~mask2].reset_index(drop=True)

In [281]:
broken_dialog = lambda snt: bool(re.match(r".*-.*", snt))

In [287]:
df_5.sample(50)

,eng_text,pol_text
500060,No.,Nie.
167515,"Now, in about two minutes.",Teraz za około dwie minuty.
122837,Who knows it'll take ten or even twenty years.,"Kto wie, może to zająć nawet dziesięć lub dwad..."
345213,"I don't know, are you?",Nie wiem. A ty?
326702,Gibbons.,Gibbons.
120673,"Personally, I don't believe either statement.","Nie wierzę ani w jedno, ani w drugie."
294643,Reading too many detective magazines.,Ma wyobraźnię.
543671,"Mr. O'Rourke, will you step forward.","Panie O'Rourke, proszę wystąpić."
229863,You don't know what a load you have taken off ...,"Nie wiesz co obciążenie, zabrałeś mi mój umysł."
540812,Very quiet.,Bardzo spokojnie.


#### 2.6 Different Last Case

In [295]:
mask = df_5['eng_text'].str[-1] != df_5['pol_text'].str[-1]
mask.sum()

57558

In [299]:
df_6 = df_5[~mask].reset_index(drop=True)

#### 2.7 Text length difference

In [310]:
mask = df_6['eng_text'].str.split
df_6['len_ratio'] = df_6['eng_text'].str.split().str.len() / df_6['pol_text'].str.split().str.len()
df_6.head()

,eng_text,pol_text,len_ratio
0,You want to call your daddy?,Chcesz zadzwonić do taty?,1.500000
1,"Yeah, I want to tell him I'm okay.","Tak, powiem, że wszystko w porządku.",1.333333
2,Okay.,Dobrze.,1.000000
3,Lizzy... Be careful of your husband.,"Lizzy, uważaj na męża.",1.500000
4,I can only lead you to the truth.,/Mogę naprowadzić cię na prawdę.,1.600000


In [338]:
thres_left = df_6["len_ratio"].quantile(0.05)
thres_right = df_6["len_ratio"].quantile(0.95)
thres_left, thres_right

(0.75, 2.5)

In [339]:
mask = (df_6["len_ratio"] >= thres_left) & (df_6["len_ratio"] <= thres_right)
mask.sum()

472209

In [341]:
df_7 = df_6[mask].reset_index(drop=True)
df_7.sample(20)

,eng_text,pol_text,len_ratio
186823,He can't take away our king.,Nie może zabrać naszego króla.,1.200000
310121,"Cherokee, did you bring the luggage?","Cherokee, masz bagaż?",2.000000
167583,Who am I?,Kim jestem?,1.500000
416941,If you can't tell me-- l can tell you.,Jeśli nie możesz mi powiedzieć...,1.800000
296404,People who have them can buy much breadfruit.....,"Ludzie, którzy je mają, mogą kupić dużo owoców...",0.909091
157628,No.,Nie.,1.000000
28342,"Tell me, he can split me in two like a jolly a...","Nie mów mi, że on potrafi mnie rozdzielić jak ...",1.200000
69039,Your Uncle Bean is sinking.,Twój wujek Bean tonie.,1.250000
108439,We've been fishing him for two years and you g...,"Próbowaliśmy go złapać przez dwa lata, a teraz...",1.000000
300845,"Colonel, it was your own idea.","Pułkowniku, to pana pomysł.",1.500000


In [342]:
df_7.sample(50)

,eng_text,pol_text,len_ratio
467522,Me too. I'm a commissioner too.,Ja też jestem komisarzem.,1.500000
97148,I can't take it anymore.,Zaczynam mieć tego wszystkiego dosyć.,1.000000
419682,He's on almost every number but the right one.,Obstawia prawie każdy numer oprócz właściwego.,1.500000
172310,Where was he shot?,Gdzie został postrzelony?,1.333333
148201,We will wait until dark now.,Teraz poczekamy do zmroku.,1.500000
212848,What's the matter with that?,Kim był Davis?,1.666667
166387,"There's a sucker Born every second, pal.",Mhm. Co sekundę rodzi się frajer.,1.166667
253827,There is beautiful.,Jest taki piękny.,1.000000
131352,How much would that cost?,IIe by to kosztowało?,1.250000
170957,Aye.,Aye.,1.000000


#### 2.8 Sentences starting with * fix

In [410]:
fix_star = lambda snt: re.sub(r"^ *[*].*", "", snt)
df_7['eng_text'] = df_7['eng_text'].apply(fix_star)
df_7['pol_text'] = df_7['pol_text'].apply(fix_star)

#### 2.9 Action Comments

In [417]:
def mask_distinct(df, col1, col2, chars):
    chars_escaped = "".join(re.escape(c) for c in chars)
    templ = f"[{chars_escaped}]"

    m1 = df[col1].str.contains(templ, regex=True, na=False)
    m2 = df[col2].str.contains(templ, regex=True, na=False)

    return m1 ^ m2

chars = '[]()/\*+#$"️'
mask = mask_distinct(df_7, "eng_text", "pol_text", chars)

df_8 = df_7[~mask].reset_index(drop=True)
df_8.shape

(467550, 3)

In [419]:
df_8.sample(50)

,eng_text,pol_text,len_ratio
251062,Mr Parry...,Panie Parry...,1.000000
368032,To the men we have loved.,"Za mężczyzn, których kochamy.",1.500000
205616,May stalin conquer.,Niech Stalin zwycięży.,1.000000
148661,"To them, this is heaven.",Dla nich to niebo.,1.250000
276247,What would you like?,Czego pan sobie życzy?,1.000000
348678,False alarm.,Fałszywy alarm.,1.000000
436840,"According to George here, Michael is anxious t...","George twierdzi, że Michael chce odejść.",1.500000
436971,My wife's lost her sense of humour.,Moja żona straciła poczucie humoru.,1.400000
41996,Give me the iodine.,Daj mi jodynę.,1.333333
225762,It rather reminds me of our coastline at home.,To przypomina mi wybrzeże w moich rodzinnych o...,1.125000


In [ ]:
mask = df_7['eng_text'].apply(lambda x: bool(re.search(r"")))

In [401]:
df_7[~mask1 & mask2]

,eng_text,pol_text,len_ratio
19960,"If you'll pardon me a minute, I'll...","* Słyszę, mówi matka * Czuję...",1.166667
49667,I'll put my foot down So shall it be,* Będę kroczył śmiało * Jak będzie tak będzie,1.000000
49668,I'm strictly on the up and up so everyone beware,* Jestem dokładnie na górze * Więc każdy niech...,0.909091
49669,We find out which one she prefers by letting h...,"* Dowiemy się, którego ona chce * Niech sama z...",1.100000
49670,If she prefers the other man the husband steps...,* Jeśli będzie chciała tego drugiego * To mąż ...,1.000000
50260,'Cause I'm coming 'round the mountain with a b...,* Bo idę poprzez góry z bandżo na kolanie,1.333333
50261,With a banjo,* Z moim bandżo,0.750000
50262,"Oh, how we'd cry for Firefly if Firefly should...",* Jak będziemy płakać za Firefly'em * Jeśli Fi...,1.000000
263063,For men must work and women must weep?,*Dla mężczyzn kobieta musi się starać i popłakać?,1.000000
263064,Or however it goes.,* Jednakże to działa.,1.000000


In [359]:
mask1 = df_7['eng_text'].str.contains("[", regex=False)
mask2 = df_7['pol_text'].str.contains("[", regex=False)

In [360]:
df_7[mask1 & ~mask2].sample(10)

,eng_text,pol_text,len_ratio
54953,"[coarsely] Are you going to talk, or do I have...","Będziesz mówił, czy mam cię do tego zmusić?",1.625000
142121,[Arguing In Italian] We can't do it.,Nic z tego.,2.333333
430824,[multiple conversations] ...we're going to thr...,Przestań pustoszyć kieszenie porządnych obywat...,1.454545
178698,"[Knocking] Mrs Callahan, may I speak to you ju...","Pani Callahan, mógłbym zamienić z panią parę s...",1.500000
374555,[Sighs] Sure feels good to take the weight off...,Poczułem się lepiej zdejmując ciężar z pani stóp.,1.375000
178373,[Laughing] Tea for Destry?,Herbaty dla Destry'ego?,1.333333
313419,"[radio] Bomb bay doors open, six degrees right...",Luk bombowy otwarty. Kurs sześć stopni w prawo.,1.250000
425377,"[ Advert playing: ] ""Ha ha, love that soap.""","""Ha-ha! Uwielbiam to mydło! .""",1.800000
179195,[Lily Belle] Don't you ever hit me!,Nie bij mnie! Przestań!,1.750000
301009,[CRASH] HEY! WATCH WHAT YOU DO.,"Uważaj, co robisz, te rzeczy kosztują.",1.000000


In [364]:
df_7.shape

(472209, 3)

In [363]:
df_7[~mask1 & mask2].shape

(15, 3)

In [365]:
df_7[~(~mask1 & mask2)].shape

(472194, 3)